# PatchGuard: Adversarial Patch Defense

Train a small window classifier and use majority voting to limit the effect of a localized patch. This is a **simplified PatchGuard-inspired exercise**, not a certified defense or a proof of robustness.

Complete the three marked TODO functions, compare clean and patched accuracy, save the model `state_dict`, and submit it. The notebook uses an offline MNIST subset: first 3,000 train and first 500 public test examples. The evaluator uses separate hidden examples and several patch locations. Compare both returned accuracy values with the thresholds in the CTFd challenge description.

The source prose says pixels outside each window are zeroed, but its code passes **cropped windows** to the CNN. The source attack is untargeted cross-entropy ascent, with a separate 12×12 patch per image, 20 steps, and step size 8/255. Its default `patch_x=patch_y=12` places the patch at `(2,2)` on a 28×28 image.

In [ ]:
from range_device import select_device, describe_device
device = select_device()
print("Compute device:", describe_device(device))

import os, random, time
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import requests
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from IPython.display import display, Markdown

SEED=20260928
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
device=select_device()
print("Training device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (CUDA unavailable in this workspace)")
torch.set_num_threads(2)
BATCH_SIZE=10
DATA_DIR=Path("/workspace/data")
MODEL_PATH=Path("/workspace/patchguard_model.pt")

In [ ]:
# Load only local files; download=False keeps the exercise usable offline.
transform=transforms.ToTensor()
train=datasets.MNIST(str(DATA_DIR), train=True, download=False, transform=transform)
test=datasets.MNIST(str(DATA_DIR), train=False, download=False, transform=transform)
train_images=torch.stack([train[i][0] for i in range(3000)]).to(device)
train_labels=torch.tensor([train[i][1] for i in range(3000)], device=device)
test_images_clean=torch.stack([test[i][0] for i in range(500)]).to(device)
test_labels=torch.tensor([test[i][1] for i in range(500)], device=device)
print("Train:", tuple(train_images.shape), "Public test:", tuple(test_images_clean.shape))

## 1. Clean baseline

This fixed LeNet-style CNN matches the source assignment. It takes either 28×28 images or 16×16 crops.

In [ ]:
class BasicCNN(nn.Module):
    def __init__(self, patch_size, in_channels, channels=(32,64), fc_dim=128, num_classes=10):
        super().__init__(); c1,c2=channels
        feat_dim=c2*(patch_size//4)*(patch_size//4)
        self.net=nn.Sequential(
            nn.Conv2d(in_channels,c1,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(c1,c2,3,padding=1), nn.ReLU(), nn.MaxPool2d(2), nn.Flatten(),
            nn.Linear(feat_dim,fc_dim), nn.ReLU(), nn.Linear(fc_dim,num_classes))
    def forward(self,x): return self.net(x)

def train_model(model,x,y,epochs):
    model.train(); opt=torch.optim.Adam(model.parameters(),lr=1e-3)
    for epoch in range(epochs):
        order=torch.randperm(x.size(0),device=x.device); total=0.0
        for start in range(0,x.size(0),BATCH_SIZE):
            idx=order[start:start+BATCH_SIZE]; loss=F.cross_entropy(model(x[idx]),y[idx])
            opt.zero_grad(); loss.backward(); opt.step(); total+=loss.item()
        print(f"Epoch {epoch+1:02d}/{epochs} | average loss {total/(x.size(0)//BATCH_SIZE):.4f}")

def ordinary_accuracy(model,x,y):
    model.eval(); correct=0
    with torch.inference_mode():
        for i in range(0,x.size(0),128): correct+=int((model(x[i:i+128]).argmax(1)==y[i:i+128]).sum())
    return correct/x.size(0)

In [ ]:
# The source uses 30 epochs. This can take several minutes on CPU.
baseline_model=BasicCNN(28,1).to(device)
start=time.perf_counter(); train_model(baseline_model,train_images,train_labels,30)
baseline_train_seconds=time.perf_counter()-start
print("Baseline clean accuracy:", round(ordinary_accuracy(baseline_model,test_images_clean,test_labels),3))
print(f"Baseline training time: {baseline_train_seconds/60:.1f} min")

## 2. Untargeted patch attack

Each image gets its own patch. To reduce memory on smaller GPUs, the 500 examples are attacked in small independent batches; this keeps the source attack settings and per-image gradients.

In [ ]:
def _attack_batch(model,x,y,patch_size,top,left,steps=20,step_size=8/255):
    base=x.detach(); patch=torch.zeros(x.size(0),x.size(1),patch_size,patch_size,device=x.device)
    area=(slice(None),slice(None),slice(top,top+patch_size),slice(left,left+patch_size))
    for _ in range(steps):
        patch.requires_grad_(True); patched=base.clone()
        patched[area]=(base[area]+patch).clamp(0,1)
        grad,=torch.autograd.grad(F.cross_entropy(model(patched),y),patch)
        with torch.no_grad(): patch=(patch+step_size*grad.sign()).clamp(-1,1)
    result=base.clone(); result[area]=(base[area]+patch).clamp(0,1)
    return result

def generate_adversarial_img(model,x,y,patch_size=12,patch_x=12,patch_y=12,steps=20,step_size=8/255):
    h,w=x.shape[-2:]; top,left=h//2-patch_y,w//2-patch_x
    if min(top,left)<0 or top+patch_size>h or left+patch_size>w: raise ValueError("Patch does not fit")
    model.eval(); parts=[]
    for i in range(0,x.size(0),25): parts.append(_attack_batch(model,x[i:i+25],y[i:i+25],patch_size,top,left,steps,step_size))
    return torch.cat(parts)

test_images_adv=generate_adversarial_img(baseline_model,test_images_clean,test_labels)
print("Baseline accuracy on patched images:",round(ordinary_accuracy(baseline_model,test_images_adv,test_labels),3))

In [ ]:
idx=10
fig,ax=plt.subplots(1,2,figsize=(6,3))
ax[0].imshow(test_images_clean[idx].detach().cpu().squeeze(),cmap="gray"); ax[0].set_title(f"Clean | label {int(test_labels[idx])}")
ax[1].imshow(test_images_adv[idx].detach().cpu().squeeze(),cmap="gray"); ax[1].set_title("Patched")
for a in ax:a.axis("off")
plt.tight_layout(); plt.show()

## 3. TODO: crop sliding windows

Return a tensor of shape `[number_of_windows, channels, window_size, window_size]` in row-major order. For 28×28, window 16, stride 2, the result has 49 crops. `Tensor.unfold` can help.

In [ ]:
def extract_windows(img,window_size,stride):
    # TODO: Implement the sliding crops in row-major order.
    raise NotImplementedError("Implement extract_windows")

In [ ]:
# Compare both values and order with this simple reference after you implement extract_windows.
def extract_windows_reference(img,window_size,stride):
    result=[]
    for top in range(0,img.shape[-2]-window_size+1,stride):
        for left in range(0,img.shape[-1]-window_size+1,stride):
            result.append(img[:,top:top+window_size,left:left+window_size])
    return torch.stack(result)

expected=extract_windows_reference(test_images_clean[0],16,2)
actual=extract_windows(test_images_clean[0],16,2)
assert actual.shape==(49,1,16,16) and torch.equal(actual,expected)
print("Window output matches the reference (49 crops).")

## 4. Train on window crops

Each crop inherits its original image label. One optimizer update uses 10 original images (490 windows), matching the source training loop.

In [ ]:
def train_pg_model(model,x,y,epochs,window_size=16,stride=2):
    model.train(); opt=torch.optim.Adam(model.parameters(),lr=1e-3); n=x.size(0)
    for epoch in range(epochs):
        order=torch.randperm(n,device=x.device); total=0.0
        for start in range(0,n,BATCH_SIZE):
            idx=order[start:start+BATCH_SIZE]; batches=[]; labels=[]
            for j in idx:
                windows=extract_windows(x[j],window_size,stride)
                batches.append(windows); labels.append(y[j].expand(windows.size(0)))
            inputs=torch.cat(batches).to(device); targets=torch.cat(labels).to(device)
            loss=F.cross_entropy(model(inputs),targets)
            opt.zero_grad(); loss.backward(); opt.step(); total+=loss.item()
        print(f"Epoch {epoch+1:03d}/{epochs} | average loss {total/(n//BATCH_SIZE):.4f}")

In [ ]:
# Keep the source's 100 epochs; this may take several minutes on CPU.
patch_model=BasicCNN(16,1).to(device)
start=time.perf_counter(); train_pg_model(patch_model,train_images,train_labels,100,16,2)
patch_model_train_seconds=time.perf_counter()-start
print(f"Window model training time: {patch_model_train_seconds/60:.1f} min")

## 5. TODO: majority vote and accuracy

Classify each window and select the most frequent class. On a tie, `argmax` selects the smallest class index. Accuracy is `correct / number_of_images`.

In [ ]:
def get_window_predictions(model,windows):
    model.eval()
    with torch.inference_mode(): return model(windows).argmax(dim=1)

def patchguard_predict(model,img,window_size=16,stride=2):
    # TODO: extract, classify, and majority-vote the windows. Return a Python int.
    raise NotImplementedError("Implement patchguard_predict")

def patchguard_accuracy(model,x,y,window_size=16,stride=2):
    model.eval(); correct=0
    for i in range(x.size(0)):
        # TODO: compare patchguard_predict(...) with y[i] and update correct.
        raise NotImplementedError("Implement patchguard_accuracy")
    return correct/x.size(0)

In [ ]:
clean_accuracy=patchguard_accuracy(patch_model,test_images_clean,test_labels,16,2)
robust_accuracy=patchguard_accuracy(patch_model,test_images_adv,test_labels,16,2)
print(f"Window model clean accuracy: {clean_accuracy:.3f}")
print(f"Window model patched accuracy: {robust_accuracy:.3f}")

## 6. Save and submit

Submit only the `state_dict`; the protected evaluator enforces the fixed architecture and tests separate hidden data. A success response includes a flag to submit to the PatchGuard CTFd challenge.

In [ ]:
torch.save(patch_model.state_dict(),MODEL_PATH)
print(f"Saved state_dict to {MODEL_PATH} ({MODEL_PATH.stat().st_size/(1024**2):.2f} MiB)")

In [ ]:
PATCHGUARD_TARGET_URL=os.environ.get("PATCHGUARD_TARGET_URL","http://patchguard-target:8000").rstrip("/")
CTFD_USER_ID=os.environ.get("CTFD_USER_ID","anonymous")
CTFD_USERNAME=os.environ.get("CTFD_USERNAME","anonymous")
CTFD_PUBLIC_URL=os.environ.get("CTFD_PUBLIC_URL","http://localhost:8001").rstrip("/")

def submit_patchguard_model(path="/workspace/patchguard_model.pt"):
    model_path=Path(path)
    if not model_path.is_file(): raise FileNotFoundError(model_path)
    with model_path.open("rb") as f:
        response=requests.post(f"{PATCHGUARD_TARGET_URL}/score",
            files={"model":("patchguard_model.pt",f,"application/octet-stream")},
            data={"user_id":CTFD_USER_ID,"username":CTFD_USERNAME},timeout=180)
    response.raise_for_status(); result=response.json()
    display(Markdown(f"**CTFd user:** {CTFD_USERNAME} (ID {CTFD_USER_ID})")); display(result)
    if result.get("success"): display(Markdown(f"Submit the returned flag to [CTFd]({CTFD_PUBLIC_URL}/challenges)."))
    return result

print("Ready. Run submit_patchguard_model() when you want to submit.")

## Reflection

- Which windows intersect the source patch at `(2,2)`?
- How does majority voting tolerate some incorrect window predictions?
- Why does the evaluator use more than one patch position?
- Compare your public metrics with the CTFd thresholds.